# Machine Learning Workflow: Linear Regression Car Valuation
End-to-end pipeline: feature transformation, train-test splitting, fitting Linear Regression, performance metrics, and model saving via pickle.

In [1]:
import os
import pickle
from datetime import datetime
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

### 1. Ingest Raw Dataset

In [2]:
data_loc = os.path.join('..', 'data', 'raw data', 'car data.csv')
raw_cars = pd.read_csv(data_loc)
raw_cars.head()

,Car_Name,Year,Selling_Price,Present_Price,Kms_Driven,Fuel_Type,Seller_Type,Transmission,Owner
0,ritz,2014,3.35,5.59,27000,Petrol,Dealer,Manual,0
1,sx4,2013,4.75,9.54,43000,Diesel,Dealer,Manual,0
2,ciaz,2017,7.25,9.85,6900,Petrol,Dealer,Manual,0
3,wagon r,2011,2.85,4.15,5200,Petrol,Dealer,Manual,0
4,swift,2014,4.60,6.87,42450,Diesel,Dealer,Manual,0


### 2. Feature Cleaning & Indicator Encoding

In [3]:
# Derive vehicle age from current calendar year
current_year = datetime.now().year
raw_cars['Age'] = current_year - raw_cars['Year']

# Remove raw identifiers
subset = raw_cars.drop(columns=['Car_Name', 'Year'])

# One-hot encode categorical features (drop first level to prevent collinearity)
processed_cars = pd.get_dummies(subset, drop_first=True, dtype=int)

# Consistent feature ordering
ordered_cols = ['Selling_Price', 'Present_Price', 'Kms_Driven', 'Owner', 'Age',
                'Fuel_Type_Diesel', 'Fuel_Type_Petrol', 'Seller_Type_Individual', 'Transmission_Manual']
processed_cars = processed_cars[ordered_cols]
processed_cars.head()

,Selling_Price,Present_Price,Kms_Driven,Owner,Age,Fuel_Type_Diesel,Fuel_Type_Petrol,Seller_Type_Individual,Transmission_Manual
0,3.35,5.59,27000,0,12,0,1,0,1
1,4.75,9.54,43000,0,13,1,0,0,1
2,7.25,9.85,6900,0,9,0,1,0,1
3,2.85,4.15,5200,0,15,0,1,0,1
4,4.60,6.87,42450,0,12,1,0,0,1


### 3. Save Processed Dataset to Disk

In [4]:
target_processed_csv = os.path.join('..', 'data', 'processed data', 'processed_data.csv')
processed_cars.to_csv(target_processed_csv, index=False)
print(f'Processed dataset successfully written to: {target_processed_csv}')

Processed dataset successfully written to: ..\data\processed data\processed_data.csv


### 4. Partition Train and Test Sets

In [5]:
X = processed_cars.drop(columns=['Selling_Price'])
y = processed_cars['Selling_Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f'Train samples: {X_train.shape[0]} | Test samples: {X_test.shape[0]}')

Train samples: 160 | Test samples: 41


### 5. Train Linear Regression Model

In [6]:
reg_model = LinearRegression()
reg_model.fit(X_train, y_train)
print('Training complete.')

Training complete.


### 6. Model Evaluation

In [7]:
y_pred = reg_model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f'R2 Score (Variance Explained): {r2:.4f}')
print(f'Mean Absolute Error (MAE)    : {mae:.4f} Lakhs')
print(f'Root Mean Squared Error (RMSE): {rmse:.4f} Lakhs')

R2 Score (Variance Explained): 0.8450
Mean Absolute Error (MAE)    : 1.4935 Lakhs
Root Mean Squared Error (RMSE): 1.9876 Lakhs


### 7. Fitted Weights (Feature Impact)

In [8]:
weights_df = pd.DataFrame({
    'Feature_Name': X.columns,
    'Weight_Value': reg_model.coef_
}).sort_values(by='Weight_Value', ascending=False)
weights_df

,Feature_Name,Weight_Value
4,Fuel_Type_Diesel,2.026251
6,Seller_Type_Individual,1.029675
0,Present_Price,0.432070
5,Fuel_Type_Petrol,0.103376
1,Kms_Driven,-0.000034
3,Age,-0.400726
2,Owner,-1.407689
7,Transmission_Manual,-2.305149


### 8. Pickle Serialization & Verification

In [9]:
model_export_path = os.path.join('..', 'models', 'lr_model.pkl')
with open(model_export_path, 'wb') as outfile:
    pickle.dump(reg_model, outfile)
print(f'Model persisted to: {model_export_path}')

# Verify reload
with open(model_export_path, 'rb') as infile:
    loaded_reg = pickle.load(infile)
sample_pred = loaded_reg.predict(X_test.iloc[:1])
print(f'Sample Test Prediction: {sample_pred[0]:.2f} Lakhs (Actual Value: {y_test.iloc[0]:.2f} Lakhs)')

Model persisted to: ..\models\lr_model.pkl
Sample Test Prediction: 7.39 Lakhs (Actual Value: 5.85 Lakhs)
